# FASE 6 — Auditoría de Modelado MVP
## Regulación de IA vs Ecosistema de IA (Boletín 16821-19 Chile)

---

**¿Qué es este documento?**
Este notebook es una **auditoría visual** de los resultados de modelado de la Fase 6.
Permite a un humano verificar paso a paso cada modelo, sus datos, y sus resultados.

**¿Qué NO hace este notebook?**
- NO recalcula ningún modelo (los modelos ya fueron entrenados por el pipeline)
- NO modifica ningún dato
- Solo LEE los CSVs generados y los presenta de forma visual

**¿Cómo leerlo?**
Cada sección sigue esta estructura:
1. **DATOS** → Qué datos se usaron (train/test)
2. **MODELO** → Qué técnica se aplicó
3. **RESULTADO** → Coeficientes, métricas, gráficos
4. **INTERPRETACIÓN** → Qué significa en lenguaje simple

---
## 1. Datos: Split Train / Test

Antes de ver cualquier modelo, es fundamental entender CÓMO se dividieron los datos.

**¿Por qué dividir?**
- **Train (entrenamiento):** Datos con los que el modelo "aprende" los patrones
- **Test (prueba):** Datos que el modelo NUNCA vio durante el entrenamiento, usados para verificar que el modelo generaliza bien

**En este proyecto:**
- **43 países** en total
- **34 países TRAIN** → El modelo aprende de estos
- **9 países TEST** → Reservados para verificación futura (Fase 7)

**Punto clave:** Chile (CHL) está en el grupo TRAIN, lo que significa que el modelo sí vio los datos de Chile durante el entrenamiento.

In [ ]:
import os
from pathlib import Path

# Find FASE6 root
cwd = Path.cwd()
candidate = cwd
for _ in range(5):
    if (candidate / "outputs" / "fase6_manifest.json").exists():
        os.chdir(candidate)
        break
    candidate = candidate.parent

import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    pass

plt.rcParams.update({"figure.figsize": (12, 6), "figure.dpi": 100})

OUTPUTS = Path("outputs")
F5_BUNDLE = Path("../FASE5/outputs/phase6_ready")

# Cargar split
split = pd.read_csv(F5_BUNDLE / "phase6_train_test_split.csv")
train = split[split["split"] == "train"]
test = split[split["split"] == "test"]

print("=" * 60)
print("SPLIT TRAIN / TEST")
print("=" * 60)
print()
print("TRAIN (34 paises) - El modelo aprende de estos:")
print("  ", ", ".join(sorted(train["iso3"].tolist())))
print()
print("TEST (9 paises) - Reservados para verificacion futura:")
print("  ", ", ".join(sorted(test["iso3"].tolist())))
print()
chile_split = split[split["iso3"] == "CHL"]["split"].values[0]
print(">>> Chile esta en:", chile_split.upper())

In [ ]:
# Visualización del split por región
fm = pd.read_csv(F5_BUNDLE / "phase6_feature_matrix.csv")
split_region = fm.groupby(["region", "split"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
split_region.plot(kind="barh", stacked=True, ax=ax, color=["#3498db", "#e74c3c"])
ax.set_xlabel("Numero de paises")
ax.set_title("Distribucion Train/Test por Region")
ax.legend(["Train", "Test"])
plt.tight_layout()
plt.show()

print()
print("NOTA: Los 9 paises TEST son 5 de Europa + 4 de Latinoamerica.")
print("Estos paises NO seran usados en el entrenamiento de los modelos.")
print("Se reservan para Fase 7 (sensitivity analysis).")

---
## 2. Variables del modelo

**¿Qué datos alimentan los modelos?**

Hay tres tipos de variables:

| Tipo | Nombre | Qué mide | Ejemplo |
|------|--------|----------|---------|
| **X1** (Tratamiento) | Variables regulatorias | Cómo regula el país la IA | nº de leyes vinculantes |
| **X2** (Controles) | Variables socioeconómicas | Factores que también afectan el ecosistema | PIB per cápita, penetración internet |
| **Y** (Resultado) | Variables de ecosistema | Lo que queremos medir | Inversión, adopción, innovación |

**La pregunta central es:** Después de controlar por X2 (factores socioeconómicos), ¿la regulación (X1) se asocia con el ecosistema (Y)?

In [ ]:
# Mostrar variables por rol
with open(F5_BUNDLE / "phase6_modeling_contract.yaml") as f:
    import yaml
    contract = yaml.safe_load(f)

roles = contract["variables_by_role"]

print("=" * 60)
print("VARIABLES POR ROL")
print("=" * 60)

print()
print("X1 - VARIABLES REGULATORIAS (Tratamiento):")
for v in roles.get("X1_regulatory", []):
    print("  -", v)

print()
print("X2 - CONTROLES SOCIOECONOMICOS (12 variables):")
for v in roles.get("X2_control", []):
    print("  -", v)

print()
print("Y - VARIABLES DE RESULTADO por sub-pregunta:")
for role, vars_list in roles.items():
    if role.startswith("Y_"):
        print(f"  {role}: {len(vars_list)} variables")
        for v in vars_list:
            print(f"    - {v}")

In [ ]:
# Cobertura de datos: qué tan completos están
key_vars = roles.get("X1_regulatory", []) + ["n_binding", "n_non_binding"]
print("=" * 60)
print("COBERTURA DE DATOS")
print("=" * 60)
print()
print("Variable | Paises con datos | Cobertura")
print("-" * 50)
for v in key_vars:
    if v in fm.columns:
        n = fm[v].notna().sum()
        print(f"  {v:35s} | {n}/43 | {n/43*100:.0f}%")

print()
print("DECISION CLAVE: Usar variables AGREGADAS (n_binding, n_non_binding)")
print("que tienen 100% cobertura (43/43) en lugar de IAPP raw (solo 18/43).")
print("Esto permite usar todos los 43 paises en los modelos.")

---
## 3. Q4 — Tipos Regulatorios (Clustering)

### ¿Qué pregunta responde?
**¿Existen grupos naturales de países según su perfil regulatorio de IA?**

### ¿Cómo funciona?
El clustering agrupa países que son similares entre sí en sus variables regulatorias.
NO necesita una variable Y; es un análisis **no supervisado**.

### ¿Qué modelo se usó?
- **K-Means:** Divide en K grupos minimizando la distancia al centroide
- **HCA (Análisis Jerárquico):** Agrupa de abajo hacia arriba, creando un dendrograma

### Datos usados:
- **N=43 países** (todos, train + test) — el clustering no entrena/testea, solo agrupa
- **14 features:** 9 one-hots IAPP + 5 agregados regulatorios

In [ ]:
sil = pd.read_csv(OUTPUTS / "q4_silhouette_scores.csv")
print("=" * 60)
print("Q4: CALIDAD DEL CLUSTERING")
print("=" * 60)
print()
print("Silhouette Score mide que tan bien definidos estan los clusters:")
print("  - Cercano a 1.0 = clusters muy definidos (bueno)")
print("  - Cercano a 0.0 = clusters ambiguos")
print("  - Negativo = paises mal asignados")
print()
for _, row in sil.iterrows():
    score = row["silhouette_score"]
    label = "BUENO" if score > 0.5 else "ACEPTABLE" if score > 0.25 else "DEBIL"
    print(f"  {row['method']:10s} ({row['scope']:12s}): {score:.3f} -> {label}")

In [ ]:
# Visualización silhouette
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2ecc71" if s > 0.5 else "#f39c12" if s > 0.25 else "#e74c3c" for s in sil["silhouette_score"]]
bars = ax.barh(
    sil["method"] + " (" + sil["scope"] + ")",
    sil["silhouette_score"],
    color=colors, edgecolor="white", linewidth=1.5
)
ax.set_xlabel("Silhouette Score")
ax.set_title("Q4: Calidad de Clustering (mayor = mejor)")
ax.axvline(x=0.5, color="green", linestyle="--", alpha=0.5, label="Bueno (>0.5)")
ax.axvline(x=0.25, color="orange", linestyle="--", alpha=0.5, label="Aceptable (>0.25)")
ax.legend()
for bar, val in zip(bars, sil["silhouette_score"]):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, "{:.3f}".format(val), va="center")
plt.tight_layout()
plt.show()

In [ ]:
# Clusters de países
clusters = pd.read_csv(OUTPUTS / "q4_clusters.csv")
print("=" * 60)
print("Q4: PAISES POR CLUSTER (K-Means, K=4)")
print("=" * 60)
print()
for c in sorted(clusters["cluster_kmeans_n43"].unique()):
    paises = clusters[clusters["cluster_kmeans_n43"] == c]["country_name_canonical"].tolist()
    iso3s = clusters[clusters["cluster_kmeans_n43"] == c]["iso3"].tolist()
    label = ", ".join(iso3s[:10])
    if len(iso3s) > 10:
        label += "..."
    print(f"  Cluster {c} ({len(paises)} paises): {label}")

chile_cluster = clusters[clusters["iso3"] == "CHL"]["cluster_kmeans_n43"].values[0]
print()
print(f">>> CHILE esta en el Cluster {chile_cluster}")
print(f"    Paises en su cluster: {', '.join(clusters[clusters['cluster_kmeans_n43'] == chile_cluster]['iso3'].tolist())}")

In [ ]:
# Heatmap de centroides
centroids = pd.read_csv(OUTPUTS / "q4_centroids.csv")
if len(centroids) > 0:
    fig, ax = plt.subplots(figsize=(14, 4))
    centroid_data = centroids.set_index("cluster_label")
    sns.heatmap(centroid_data, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax, linewidths=0.5)
    ax.set_title("Q4: Perfil promedio de cada cluster (centroides)")
    ax.set_xlabel("Variables regulatorias")
    ax.set_ylabel("Cluster")
    plt.tight_layout()
    plt.show()
    print()
    print("INTERPRETACION: Cada fila es un cluster. Los valores muestran")
    print("el promedio de cada variable regulatoria en ese cluster.")

---
## 4. Q1 — Inversión en Ecosistema IA

### ¿Qué pregunta responde?
**¿La regulación de IA se asocia con MENOR inversión en tecnología emergente?**

### Modelo: OLS (Mínimos Cuadrados Ordinarios)
- **Entrenamiento:** Se ajusta una recta: Y = β₀ + β₁·X1 + β₂·X2 + ... + ε
- **β (coeficiente):** Si β es negativo → más regulación = menos inversión
- **IC95 (Intervalo de Confianza):** Rango donde está el "verdadero" β con 95% de certeza
- **Si IC95 NO cruza el 0:** La asociación es estadísticamente significativa

### Datos:
- **N=36 países** (con datos completos de inversión + regulación agregada)
- **Y (6 variables de inversión):** investment emerging tech, AI unicorns, VC availability, etc.
- **X1:** n_binding, n_non_binding, n_hybrid, regulatory_intensity, n_regulatory_mechanisms
- **X2:** 12 controles socioeconómicos

### Validación:
- **Bootstrap 2000:** Se re-entrena el modelo 2000 veces con muestras aleatorias para calcular IC95 robustos
- **Ridge/Lasso:** Modelos alternativos que controlan colinealidad
- **FDR Benjamini-Hochberg:** Corrección para comparaciones múltiples

In [ ]:
q1 = pd.read_csv(OUTPUTS / "q1_results.csv")
q1_ols = q1[(q1["model"] == "OLS_full") & (q1["x_var"] != "const") & q1["coefficient"].notna()].copy()

focal = "n_binding_z" if "n_binding_z" in q1_ols["x_var"].unique() else "n_binding"
focal_rows = q1_ols[q1_ols["x_var"] == focal].copy()
focal_rows = focal_rows.sort_values("coefficient")

print("=" * 60)
print("Q1: RESULTADOS — Asociacion entre regulacion e inversion")
print("=" * 60)
print()
print("Variable X1: n_binding (numero de leyes vinculantes de IA)")
print("Cada punto es una variable de INVERSION diferente")
print()
print("INTERPRETACION DEL GRAFICO:")
print("  - Punto a la IZQUIERDA de la linea roja = regulacion asociada con MENOS inversion")
print("  - Punto a la DERECHA de la linea roja = regulacion asociada con MAS inversion")
print("  - Si la barra NO toca la linea roja = asociacion estadisticamente significativa")
print()

if len(focal_rows) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    y_pos = range(len(focal_rows))

    ax.errorbar(
        focal_rows["coefficient"], y_pos,
        xerr=[focal_rows["coefficient"] - focal_rows["ci95_lower"],
              focal_rows["ci95_upper"] - focal_rows["coefficient"]],
        fmt="o", color="#3498db", ecolor="#3498db", elinewidth=2, capsize=5, markersize=8
    )

    ax.axvline(x=0, color="red", linestyle="--", alpha=0.7, label="Sin asociacion (0)")
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([y[:45] for y in focal_rows["y_var"]], fontsize=9)
    ax.set_xlabel("Coeficiente beta (asociacion con n_binding)")
    n_eff = focal_rows["n_effective"].iloc[0]
    ax.set_title("Q1: Forest Plot — Regulacion vs Inversion (N=" + str(n_eff) + ")")

    for i, (_, row) in enumerate(focal_rows.iterrows()):
        p = row.get("p_value", 1)
        if p < 0.001:
            sig = "***"
        elif p < 0.01:
            sig = "**"
        elif p < 0.05:
            sig = "*"
        else:
            sig = "ns"
        ax.text(row["ci95_upper"] + 0.02, i, sig, va="center", fontsize=10, fontweight="bold")

    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Tabla detallada
print("=" * 60)
print("Q1: TABLA DE COEFICIENTES DETALLADA")
print("=" * 60)
print()
print("Columnas:")
print("  - Coef (beta): Tamaño de la asociacion")
print("  - IC95 inf/sup: Intervalo de confianza 95% (bootstrap 2000)")
print("  - p-valor: Probabilidad de ver este resultado por azar")
print("  - N: Numero de paises usados")
print("  - R2: Que tan bien explica el modelo la variable Y (0-1)")
print()

tabla = focal_rows[["y_var", "coefficient", "ci95_lower", "ci95_upper", "p_value", "n_effective", "r2"]].copy()
tabla.columns = ["Variable Y", "Coef beta", "IC95 inf", "IC95 sup", "p-valor", "N", "R2"]
display(tabla.round(4))

print()
print("SIGNIFICANCIA: * p<0.05, ** p<0.01, *** p<0.001, ns=no significativo")

In [ ]:
# Consistencia de signo
q1_cons = pd.read_csv(OUTPUTS / "q1_consistency.csv")
print("=" * 60)
print("Q1: CONSISTENCIA DE SIGNO")
print("=" * 60)
print()
print("Pregunta: Cuando n_binding aumenta, TODAS las variables de")
print("inversion se mueven en la misma direccion?")
print()
for _, row in q1_cons.iterrows():
    direction = row.get("direction_summary", "N/A")
    n_neg = row.get("n_y_negative", 0)
    n_pos = row.get("n_y_positive", 0)
    n_sig = row.get("n_y_significant_05", 0)
    print(f"  Variable X1: {row['x_var']}")
    print(f"    Y negativos: {n_neg}, Y positivos: {n_pos}, Significativos: {n_sig}")
    print(f"    Direccion: {direction}")
    print()

In [ ]:
# PSM Exploratorio
psm = pd.read_csv(OUTPUTS / "q1_psm_matched_pairs.csv")
print("=" * 60)
print("Q1: PROPENSITY SCORE MATCHING (Exploratorio)")
print("=" * 60)
print()
print("PSM compara paises CON ley IA vs SIN ley IA,")
print("emparejados por caracteristicas socioeconomicas similares.")
print()
if "status" in psm.columns and psm["status"].iloc[0] == "insufficient_n":
    print("RESULTADO: N insuficiente para PSM.")
    print("Solo 6 paises tienen ley IA vigente -> muy pocos pares posibles.")
    print("PSM queda como 'exploratorio' - NO es concluyente.")
else:
    display(psm)

psm_bal = pd.read_csv(OUTPUTS / "q1_psm_balance_diagnostics.csv")
if "smd_pre" in psm_bal.columns and psm_bal["smd_pre"].notna().any():
    print()
    print("BALANCE DE COVARIABLES (Standardized Mean Difference):")
    print("  SMD < 0.1 = buen balance")
    print("  SMD > 0.2 = desbalance preocupante")
    fig, ax = plt.subplots(figsize=(10, 6))
    x = range(len(psm_bal))
    ax.barh(x, psm_bal["smd_pre"], alpha=0.5, label="Pre-matching", color="#e74c3c")
    if "smd_post" in psm_bal.columns and psm_bal["smd_post"].notna().any():
        ax.barh(x, psm_bal["smd_post"], alpha=0.5, label="Post-matching", color="#2ecc71")
    ax.set_yticks(list(x))
    ax.set_yticklabels(psm_bal["variable"], fontsize=8)
    ax.axvline(x=0.1, color="gray", linestyle="--", alpha=0.5)
    ax.axvline(x=-0.1, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("SMD")
    ax.set_title("Q1 PSM: Balance de covariables")
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 5. Q3 — Innovación

### ¿Qué pregunta responde?
**¿La regulación de IA se asocia con menor innovación tecnológica?**

### Modelo: OLS + Ridge + Gradient Boosting
- **OLS:** Regresión lineal estándar
- **Ridge:** Regresión con penalización (controla colinealidad)
- **Gradient Boosting:** Modelo no lineal basado en árboles de decisión

### Datos:
- **Primary tier (N=40-42):** 6 variables de innovación con buena cobertura
- **Auxiliary tier (N=26):** Stanford publications (pocos datos → se analiza aparte)

### Validación:
- **5-fold CV repetida (10 veces):** El modelo se entrena 50 veces con diferentes subconjuntos
- **LOOCV (Leave-One-Out):** Se entrena 42 veces, cada vez dejando 1 país fuera

In [ ]:
q3 = pd.read_csv(OUTPUTS / "q3_results.csv")

focal = "n_binding_z" if "n_binding_z" in q3["x_var"].unique() else "n_binding"
q3_primary = q3[(q3["model"] == "OLS_full") & (q3["x_var"] == focal) & (q3["tier"] == "primary") & q3["coefficient_or_importance"].notna()].copy()
q3_primary = q3_primary.sort_values("coefficient_or_importance")

print("=" * 60)
print("Q3: RESULTADOS — Regulacion vs Innovacion (Primary tier)")
print("=" * 60)
print()
print("Mismo analisis que Q1 pero con variables de INNOVACION")
print()

if len(q3_primary) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    y_pos = range(len(q3_primary))
    has_ci = q3_primary["ci95_lower"].notna().any()
    if has_ci:
        ax.errorbar(
            q3_primary["coefficient_or_importance"], y_pos,
            xerr=[q3_primary["coefficient_or_importance"] - q3_primary["ci95_lower"],
                  q3_primary["ci95_upper"] - q3_primary["coefficient_or_importance"]],
            fmt="o", color="#9b59b6", ecolor="#9b59b6", elinewidth=2, capsize=5, markersize=8
        )
    ax.axvline(x=0, color="red", linestyle="--", alpha=0.7, label="Sin asociacion (0)")
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([y[:40] for y in q3_primary["y_var"]], fontsize=9)
    ax.set_xlabel("Coeficiente beta")
    n_eff = q3_primary["n_effective"].iloc[0]
    ax.set_title("Q3: Forest Plot — Regulacion vs Innovacion (N=" + str(n_eff) + ")")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
q3_cons = pd.read_csv(OUTPUTS / "q3_consistency.csv")
print("=" * 60)
print("Q3: CONSISTENCIA")
print("=" * 60)
print()
for _, row in q3_cons.iterrows():
    tier = row.get("tier", "N/A")
    n_neg = row.get("n_y_negative", 0)
    n_pos = row.get("n_y_positive", 0)
    n_sig = row.get("n_y_significant_05", 0)
    print(f"  Tier: {tier}")
    print(f"    Y negativos: {n_neg}, Y positivos: {n_pos}, Significativos: {n_sig}")
    print()

---
## 6. Q2 — Adopción Empresarial de IA

### ¿Qué pregunta responde?
**¿La regulación de IA se asocia con menor adopción de IA por empresas?**

### Modelo: Logistic Regression + Random Forest (CLASIFICACIÓN)
A diferencia de Q1 y Q3 (regresión), aquí el modelo CLASIFICA:
- **Y=1:** País con ALTA adopción IA (sobre la mediana)
- **Y=0:** País con BAJA adopción IA (bajo la mediana)

### ¿Cómo se evalúa?
- **AUC (Area Under Curve):** Mide qué tan bien el modelo separa alta vs baja adopción
  - AUC = 0.5 → el modelo no discrimina (azar)
  - AUC = 0.7 → aceptable
  - AUC = 0.8 → bueno
  - AUC > 0.85 → excelente

### Datos:
- **7 variables Y** de adopción empresarial
- **N=20-41** dependiendo de la variable

In [ ]:
q2 = pd.read_csv(OUTPUTS / "q2_results.csv")
q2_meta = q2[q2["x_var"] == "__model_metadata__"].copy()

print("=" * 60)
print("Q2: METRICAS DE CLASIFICACION")
print("=" * 60)
print()
print("AUC = Area Under the ROC Curve")
print("  0.5 = azar | 0.7 = aceptable | 0.8 = bueno | >0.85 = excelente")
print()

cols_show = ["y_var", "model", "auc_cv_5fold_mean", "auc_cv_5fold_std", "auc_loocv", "n_effective"]
available = [c for c in cols_show if c in q2_meta.columns]
if len(available) > 0:
    display(q2_meta[available].round(3))

fig, ax = plt.subplots(figsize=(12, 6))
for model_name, color, marker in [("Logistic", "#3498db", "o"), ("RandomForest", "#e67e22", "s")]:
    subset = q2_meta[q2_meta["model"] == model_name]
    if len(subset) > 0 and "auc_cv_5fold_mean" in subset.columns:
        ax.scatter(range(len(subset)), subset["auc_cv_5fold_mean"], s=100, c=color, marker=marker, label=model_name, zorder=5)
        if "auc_cv_5fold_std" in subset.columns:
            ax.errorbar(range(len(subset)), subset["auc_cv_5fold_mean"], yerr=subset["auc_cv_5fold_std"], fmt="none", ecolor=color, elinewidth=1.5, capsize=4, alpha=0.5)

ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.5, label="Azar (0.5)")
ax.axhline(y=0.7, color="green", linestyle="--", alpha=0.3, label="Aceptable (0.7)")
y_vars = q2_meta["y_var"].unique()
ax.set_xticks(range(len(y_vars)))
ax.set_xticklabels([y[:25] for y in y_vars], rotation=45, ha="right", fontsize=9)
ax.set_ylabel("AUC")
ax.set_title("Q2: Capacidad predictiva de la regulacion sobre adopcion empresarial")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Predicciones por país
q2_pred = pd.read_csv(OUTPUTS / "q2_predictions_per_country.csv")
if len(q2_pred) > 0:
    pivot = q2_pred.pivot_table(index="iso3", columns="y_var", values="p_high_adoption")
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", center=0.5, ax=ax, linewidths=0.5)
    ax.set_title("Q2: P(alta adopcion IA) por pais")
    ax.set_xlabel("Variable de adopcion")
    ax.set_ylabel("Pais")
    plt.tight_layout()
    plt.show()

    chile_pred = q2_pred[q2_pred["iso3"] == "CHL"]
    if len(chile_pred) > 0:
        print()
        print("CHILE:")
        for _, r in chile_pred.iterrows():
            print(f"  {r['y_var']}: P(alta adopcion) = {r['p_high_adoption']:.3f}")

---
## 7. Q5 — Uso de IA por la Población

### ¿Qué pregunta responde?
**¿La regulación de IA se asocia con el uso de IA por parte de la ciudadanía?**

### Modelo: OLS + Logistic (mismo patrón que Q1 + Q2)

### Datos:
- **3 variables Y:** Anthropic usage, Anthropic collaboration, Oxford emerging tech
- **Nota:** Estas variables se reusan de Q2 (Decisión M del plan)
- Cada modelo se entrena INDEPENDIENTEMENTE

In [ ]:
q5 = pd.read_csv(OUTPUTS / "q5_results.csv")
q5_ols = q5[(q5["model"] == "OLS_full") & (q5["x_var"] != "const") & q5["coefficient"].notna()].copy()

focal = "n_binding_z" if "n_binding_z" in q5_ols["x_var"].unique() else "n_binding"
q5_focal = q5_ols[q5_ols["x_var"] == focal].sort_values("coefficient")

print("=" * 60)
print("Q5: RESULTADOS — Regulacion vs Uso IA poblacion")
print("=" * 60)
print()

if len(q5_focal) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    y_pos = range(len(q5_focal))
    has_ci = q5_focal["ci95_lower"].notna().any()
    if has_ci:
        ax.errorbar(
            q5_focal["coefficient"], y_pos,
            xerr=[q5_focal["coefficient"] - q5_focal["ci95_lower"],
                  q5_focal["ci95_upper"] - q5_focal["coefficient"]],
            fmt="o", color="#1abc9c", ecolor="#1abc9c", elinewidth=2, capsize=5, markersize=10
        )
    ax.axvline(x=0, color="red", linestyle="--", alpha=0.7)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([y[:35] for y in q5_focal["y_var"]], fontsize=10)
    ax.set_xlabel("Coeficiente beta")
    n_eff = q5_focal["n_effective"].iloc[0]
    ax.set_title("Q5: Regulacion vs Uso IA Poblacion (N=" + str(n_eff) + ")")
    plt.tight_layout()
    plt.show()

q5_cons = pd.read_csv(OUTPUTS / "q5_consistency.csv")
print()
print("CONSISTENCIA Q5:")
for _, row in q5_cons.iterrows():
    print(f"  {row['x_var']}: {row.get('direction_summary', 'N/A')} (neg:{row.get('n_y_negative',0)} pos:{row.get('n_y_positive',0)} sig:{row.get('n_y_significant_05',0)})")

---
## 8. Q6 — Uso de IA en Sector Público

### ¿Qué pregunta responde?
**¿La regulación de IA se asocia con el uso de IA en el gobierno?**

### Modelo: OLS + Logistic

### Datos:
- **Primary tier (5 Y, N=43):** Oxford public sector, e-government, digital policy, data governance, OECD INDIGO
- **Auxiliary tier (1 Y, N=31):** OECD digital gov overall (menos datos → tier separado)

In [ ]:
q6 = pd.read_csv(OUTPUTS / "q6_results.csv")

focal = "n_binding_z" if "n_binding_z" in q6["x_var"].unique() else "n_binding"
q6_primary = q6[(q6["model"] == "OLS_full") & (q6["x_var"] == focal) & (q6["tier"] == "primary") & q6["coefficient"].notna()].copy()
q6_primary = q6_primary.sort_values("coefficient")

print("=" * 60)
print("Q6: RESULTADOS — Regulacion vs Uso IA Sector Publico")
print("=" * 60)
print()

if len(q6_primary) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    y_pos = range(len(q6_primary))
    has_ci = q6_primary["ci95_lower"].notna().any()
    if has_ci:
        ax.errorbar(
            q6_primary["coefficient"], y_pos,
            xerr=[q6_primary["coefficient"] - q6_primary["ci95_lower"],
                  q6_primary["ci95_upper"] - q6_primary["coefficient"]],
            fmt="o", color="#e74c3c", ecolor="#e74c3c", elinewidth=2, capsize=5, markersize=8
        )
    ax.axvline(x=0, color="red", linestyle="--", alpha=0.7)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels([y[:35] for y in q6_primary["y_var"]], fontsize=9)
    ax.set_xlabel("Coeficiente beta")
    n_eff = q6_primary["n_effective"].iloc[0]
    ax.set_title("Q6: Regulacion vs Uso IA Sector Publico (N=" + str(n_eff) + ")")
    plt.tight_layout()
    plt.show()

q6_cons = pd.read_csv(OUTPUTS / "q6_consistency.csv")
print()
print("CONSISTENCIA Q6 (primary tier):")
for _, row in q6_cons.iterrows():
    print(f"  {row['x_var']}: neg:{row.get('n_y_negative',0)} pos:{row.get('n_y_positive',0)} sig:{row.get('n_y_significant_05',0)}")

In [ ]:
# Chile vs peers
q6_pred = pd.read_csv(OUTPUTS / "q6_predictions_per_country.csv")
if len(q6_pred) > 0:
    peers = ["CHL", "SGP", "EST", "IRL", "KOR", "URY"]
    chile_peers = q6_pred[(q6_pred["iso3"].isin(peers)) & (q6_pred["tier"] == "primary")]
    if len(chile_peers) > 0:
        print("=" * 60)
        print("Q6: CHILE vs PEERS — Uso IA Sector Publico")
        print("=" * 60)
        print()
        pivot_peers = chile_peers.pivot_table(index="iso3", columns="y_var", values="p_high_public_sector_use")
        display(pivot_peers.round(3))

---
## 9. Síntesis Preliminar

**ADVERTENCIA:** Esta síntesis usa lenguaje de **asociación**, no causalidad.
Los resultados son preliminares y requieren Fase 7 (sensitivity analysis) antes de conclusiones definitivas.

**Lo que sabemos hasta ahora:**
- Se entrenaron modelos para 6 sub-preguntas (Q1-Q6)
- Se usaron 43 países con 12 controles socioeconómicos
- Los modelos son internamente validados con CV y bootstrap
- Los 9 países TEST están reservados para Fase 7

In [ ]:
print("=" * 60)
print("SINTESIS: ASOCIACION DE REGULACION CON ECOSISTEMA")
print("=" * 60)
print()
print("n_binding = numero de leyes vinculantes de IA en el pais")
print()

summary_rows = []
for q_name, q_file, tier_filter in [
    ("Q1 Inversion", "q1_consistency.csv", None),
    ("Q3 Innovacion", "q3_consistency.csv", "primary"),
    ("Q5 Uso poblacion", "q5_consistency.csv", None),
    ("Q6 Sector publico", "q6_consistency.csv", "primary"),
]:
    try:
        cons = pd.read_csv(OUTPUTS / q_file)
        if tier_filter and "tier" in cons.columns:
            cons = cons[cons["tier"] == tier_filter]
        if len(cons) > 0:
            row = cons.iloc[0]
            summary_rows.append({
                "Sub-pregunta": q_name,
                "Direccion": row.get("direction_summary", "N/A"),
                "Y negativos": row.get("n_y_negative", "N/A"),
                "Y positivos": row.get("n_y_positive", "N/A"),
                "Sig. (p<0.05)": row.get("n_y_significant_05", "N/A"),
            })
    except Exception:
        pass

if summary_rows:
    display(pd.DataFrame(summary_rows))

print()
print("DECISIONES TECNICAS CLAVE:")
print("  - Bootstrap 2000 iteraciones para IC95")
print("  - FDR Benjamini-Hochberg para correccion multiple")
print("  - Hyperparams fijos (no tuneados)")
print("  - PCA excluido del scope")
print("  - Lenguaje 'asociacion' por defecto")
print()
print("PROXIMOS PASOS:")
print("  - Fase 7: Sensitivity analysis (excluir USA, AI leaders)")
print("  - Fase 7: Comparacion con baseline (solo controles)")
print("  - Fase 7: Hyperparameter robustness check")
print("  - Fase 8: Reporte ejecutivo + Excel final")

In [ ]:
# Verificación final
with open(OUTPUTS / "fase6_manifest.json") as f:
    manifest = json.load(f)

print("=" * 60)
print("VERIFICACION FINAL")
print("=" * 60)
print()
print("Outputs generados:", len(manifest["outputs"]))
print("Seed:", manifest["seed"])
print("Bootstrap:", manifest["n_bootstrap"])
print("Decisiones aplicadas:", len(manifest["decisions_log"]))
print()
print("Archivos:")
for f in sorted(manifest["outputs"].keys()):
    print("  OK", f)
print()
print("Este notebook NO recalculo ningun modelo.")
print("Solo lectura de CSVs para auditoria humana.")